In [64]:
import numpy as np
import math

In [65]:
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1.0 / (1.0 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [39]:
class LSTMCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        combined = input_size + hidden_size

        def init_weight(rows, cols):
            std = 1.0 / math.sqrt(hidden_size)
            return np.random.randn(rows, cols) * std

        def init_bias(size):
            return np.zeros(size)

        self.W_f = init_weight(hidden_size, combined)
        self.b_f = init_bias(hidden_size)
        
        self.W_i = init_weight(hidden_size, combined) 
        self.b_i = init_bias(hidden_size)
        
        self.W_k = init_weight(hidden_size, combined)
        self.b_k = init_bias(hidden_size)
        
        self.W_o = init_weight(hidden_size, combined)
        self.b_o = init_bias(hidden_size)

        self.zero_grad()

    def zero_grad(self):
        self.dW_f, self.db_f = np.zeros_like(self.W_f), np.zeros_like(self.b_f)
        self.dW_i, self.db_i = np.zeros_like(self.W_i), np.zeros_like(self.b_i)
        self.dW_k, self.db_k = np.zeros_like(self.W_k), np.zeros_like(self.b_k)
        self.dW_o, self.db_o = np.zeros_like(self.W_o), np.zeros_like(self.b_o)

    def forward(self, x, h_prev, c_prev):
        hx = np.concatenate([h_prev, x], axis=1)

        f = sigmoid(hx @ self.W_f.T + self.b_f)
        i = sigmoid(hx @ self.W_i.T + self.b_i)
        k = tanh(hx @ self.W_k.T + self.b_k)
        o = sigmoid(hx @ self.W_o.T + self.b_o)

        c_next = f * c_prev + i * k
        h_next = o * tanh(c_next)

        cache = (hx, h_prev, c_prev, f, i, k, o, c_next)
        return h_next, c_next, cache

    def backward(self, dh_next, dc_next, cache):
        hx, h_prev, c_prev, f, i, k, o, c_next = cache

        # 1. Gradients from hidden state
        tanh_c = tanh(c_next)
        do = dh_next * tanh_c
        dc_total = dh_next * o * (1 - tanh_c**2) + dc_next

        # 2. Gradients from cell state
        df = dc_total * c_prev
        di = dc_total * k
        dk = dc_total * i
        dc_prev = dc_total * f

        # 3. Push through activation functions (Chain Rule)
        df_raw = df * f * (1 - f)
        di_raw = di * i * (1 - i)
        do_raw = do * o * (1 - o)
        dk_raw = dk * (1 - k**2)

        # 4. Gradients for Weights
        self.dW_f += df_raw.T @ hx
        self.db_f += np.sum(df_raw, axis=0)
        self.dW_i += di_raw.T @ hx
        self.db_i += np.sum(di_raw, axis=0)
        self.dW_k += dk_raw.T @ hx
        self.db_k += np.sum(dk_raw, axis=0)
        self.dW_o += do_raw.T @ hx
        self.db_o += np.sum(do_raw, axis=0)

        # 5. Gradient to pass back to previous time step
        dhx = (df_raw @ self.W_f + di_raw @ self.W_i + 
               do_raw @ self.W_o + dk_raw @ self.W_k)

        dh_prev = dhx[:, :self.hidden_size]
        dx = dhx[:, self.hidden_size:]

        return dh_prev, dc_prev, dx

In [40]:
class NumpyEncoder:
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        self.cell = NumpyLSTMCell(input_size, hidden_size)

    def forward(self, X):
        batch_size, seq_len, _ = X.shape
        h = np.zeros((batch_size, self.hidden_size))
        c = np.zeros((batch_size, self.hidden_size))
        self.caches = []

        for t in range(seq_len):
            x_t = X[:, t, :]
            h, c, cache = self.cell.forward(x_t, h, c)
            self.caches.append(cache)

        return h, c 

    def backward(self, dh_T, dc_T):
        dh, dc = dh_T, dc_T
        dx_seq = []
        for t in reversed(range(len(self.caches))):
            dh, dc, dx = self.cell.backward(dh, dc, self.caches[t])
            dx_seq.append(dx)
            
        dx_seq.reverse()
        return np.stack(dx_seq, axis=1)



In [41]:

class NumpyDecoder:
    def __init__(self, input_size, hidden_size, vocab_size):
        self.hidden_size = hidden_size
        self.cell = NumpyLSTMCell(input_size, hidden_size)
        
        # Output layer for vocabulary prediction
        self.W_fc = np.random.randn(vocab_size, hidden_size) * 0.1
        self.b_fc = np.zeros(vocab_size)
        self.zero_grad()

    def zero_grad(self):
        self.cell.zero_grad()
        self.dW_fc = np.zeros_like(self.W_fc)
        self.db_fc = np.zeros_like(self.b_fc)

    def forward(self, Y_embeddings, h, c):
        batch_size, target_seq_len, _ = Y_embeddings.shape
        self.caches = []
        outputs = []

        for t in range(target_seq_len):
            y_t = Y_embeddings[:, t, :]
            h, c, cache = self.cell.forward(y_t, h, c)
            logits = h @ self.W_fc.T + self.b_fc
            outputs.append(logits)
            self.caches.append((cache, h))

        return np.stack(outputs, axis=1), h, c

    def backward(self, d_outputs):
        batch_size, target_seq_len, _ = d_outputs.shape
        dh_next = np.zeros((batch_size, self.hidden_size))
        dc_next = np.zeros((batch_size, self.hidden_size))
        dx_seq = []

        for t in reversed(range(target_seq_len)):
            cache, h = self.caches[t]
            dy = d_outputs[:, t, :] 
            
            # Gradients for the Fully Connected Layer
            self.dW_fc += dy.T @ h
            self.db_fc += np.sum(dy, axis=0)
            
            # Backprop through LSTM
            dh_from_fc = dy @ self.W_fc
            dh_total = dh_from_fc + dh_next
            dh_next, dc_next, dx = self.cell.backward(dh_total, dc_next, cache)
            dx_seq.append(dx)

        dx_seq.reverse()
        return dh_next, dc_next, np.stack(dx_seq, axis=1)

In [42]:
class NumpySeq2Seq:
    def __init__(self, encoder, decoder):
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, X, Y_embeddings):
        context_h, context_c = self.encoder.forward(X)
        outputs, _, _ = self.decoder.forward(Y_embeddings, context_h, context_c)
        return outputs

    def backward(self, d_outputs):
        d_context_h, d_context_c, dx_dec = self.decoder.backward(d_outputs)
        dx_enc = self.encoder.backward(d_context_h, d_context_c)
        return dx_enc, dx_dec

    def zero_grad(self):
        self.encoder.cell.zero_grad()
        self.decoder.zero_grad()


In [43]:

def cross_entropy_loss(logits, targets):
    probs = softmax(logits)
    batch_size, seq_len, _ = logits.shape
    loss = 0
    for b in range(batch_size):
        for t in range(seq_len):
            loss -= np.log(probs[b, t, targets[b, t]] + 1e-9)
    return loss / (batch_size * seq_len)

def cross_entropy_grad(logits, targets):
    probs = softmax(logits)
    batch_size, seq_len, _ = logits.shape
    d_logits = probs.copy()
    for b in range(batch_size):
        for t in range(seq_len):
            d_logits[b, t, targets[b, t]] -= 1
    return d_logits / (batch_size * seq_len)

In [44]:
class NumpyEmbedding:
    def __init__(self, vocab_size, embed_dim):
        self.W = np.random.randn(vocab_size, embed_dim) * 0.1
        self.dW = np.zeros_like(self.W)

    def forward(self, indices):
        return self.W[indices]

    def backward(self, dx, indices):
        for b in range(indices.shape[0]):
            for t in range(indices.shape[1]):
                self.dW[indices[b, t]] += dx[b, t]

    def zero_grad(self):
        self.dW.fill(0.0)

In [46]:
class Vocab:
    def __init__(self):
        self.w2i = {"<sos>": 0, "<eos>": 1}
        self.i2w = {0: "<sos>", 1: "<eos>"}
        self.n_words = 2
        
    def add(self, sentence):
        for word in sentence.split():
            if word not in self.w2i:
                self.w2i[word] = self.n_words
                self.i2w[self.n_words] = word
                self.n_words += 1


In [47]:
pairs = [
    ("i am happy", "main khush hoon"),
    ("you are good", "tum achay ho"),
    ("he is a student", "woh aik talib ilm hai"),
    ("she is happy", "woh khush hai"),
    ("i love you", "main tum se mohabbat karta hoon"),
    ("we are students", "hum talib ilm hain"),
    ("they are playing", "woh khel rahay hain"),
    ("i am learning ai", "main ai seekh raha hoon"),
    ("this is a book", "yeh aik kitab hai"),
    ("that is my house", "woh mera ghar hai"),
]



eng_vocab, urd_vocab = Vocab(), Vocab()

for eng, urd in pairs:
    eng_vocab.add(eng)
    urd_vocab.add(urd)

def get_tensors(eng_sent, urd_sent):
    x_idx = np.array([[eng_vocab.w2i[w] for w in eng_sent.split()] + [1]]) 
    urd_words = [urd_vocab.w2i[w] for w in urd_sent.split()]
    y_input_idx = np.array([[0] + urd_words]) 
    y_target_idx = np.array([urd_words + [1]]) 
    return x_idx, y_input_idx, y_target_idx


In [60]:

embed_dim = 16
hidden_size = 32


learning_rate = 0.05
epochs = 1000

enc_embed = NumpyEmbedding(eng_vocab.n_words, embed_dim)
dec_embed = NumpyEmbedding(urd_vocab.n_words, embed_dim)

encoder = NumpyEncoder(embed_dim, hidden_size)
decoder = NumpyDecoder(embed_dim, hidden_size, urd_vocab.n_words)
seq2seq = NumpySeq2Seq(encoder, decoder)

def update_weights(cell, lr):
    cell.W_f -= lr * cell.dW_f; cell.b_f -= lr * cell.db_f
    cell.W_i -= lr * cell.dW_i; cell.b_i -= lr * cell.db_i
    cell.W_k -= lr * cell.dW_k; cell.b_k -= lr * cell.db_k
    cell.W_o -= lr * cell.dW_o; cell.b_o -= lr * cell.db_o


for epoch in range(epochs):
    total_loss = 0
    for eng, urd in pairs:
        x_idx, y_input_idx, y_target_idx = get_tensors(eng, urd)
        
        seq2seq.zero_grad()
        enc_embed.zero_grad()
        dec_embed.zero_grad()
        
        # Forward
        X = enc_embed.forward(x_idx)
        Y_emb = dec_embed.forward(y_input_idx)
        logits = seq2seq.forward(X, Y_emb)
        
        loss = cross_entropy_loss(logits, y_target_idx)
        total_loss += loss
        
        # Backward
        d_logits = cross_entropy_grad(logits, y_target_idx)
        dx_enc, dx_dec = seq2seq.backward(d_logits)
        enc_embed.backward(dx_enc, x_idx)
        dec_embed.backward(dx_dec, y_input_idx)
        
        # Update
        update_weights(encoder.cell, learning_rate)
        update_weights(decoder.cell, learning_rate)
        decoder.W_fc -= learning_rate * decoder.dW_fc
        decoder.b_fc -= learning_rate * decoder.db_fc
        enc_embed.W -= learning_rate * enc_embed.dW
        dec_embed.W -= learning_rate * dec_embed.dW
        
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Average Loss: {total_loss/len(pairs):.4f}")



Epoch 50/1000 | Average Loss: 2.8377
Epoch 100/1000 | Average Loss: 2.3925
Epoch 150/1000 | Average Loss: 2.0213
Epoch 200/1000 | Average Loss: 1.7030
Epoch 250/1000 | Average Loss: 1.3196
Epoch 300/1000 | Average Loss: 0.9592
Epoch 350/1000 | Average Loss: 0.7281
Epoch 400/1000 | Average Loss: 0.5490
Epoch 450/1000 | Average Loss: 0.4147
Epoch 500/1000 | Average Loss: 0.3154
Epoch 550/1000 | Average Loss: 0.2326
Epoch 600/1000 | Average Loss: 0.1645
Epoch 650/1000 | Average Loss: 0.1191
Epoch 700/1000 | Average Loss: 0.0847
Epoch 750/1000 | Average Loss: 0.0617
Epoch 800/1000 | Average Loss: 0.0474
Epoch 850/1000 | Average Loss: 0.0382
Epoch 900/1000 | Average Loss: 0.0319
Epoch 950/1000 | Average Loss: 0.0273
Epoch 1000/1000 | Average Loss: 0.0239


In [ ]:
def translate(sentence, max_length=10):
    x_idx = np.array([[eng_vocab.w2i[w] for w in sentence.split()] + [1]])
    X = enc_embed.forward(x_idx)
    h, c = seq2seq.encoder.forward(X)
    
    current_word_idx = np.array([[0]]) 
    translated_words = []
    
    for _ in range(max_length):
        y_emb = dec_embed.forward(current_word_idx)
        h, c, _ = seq2seq.decoder.cell.forward(y_emb[:, 0, :], h, c)
        logits = h @ seq2seq.decoder.W_fc.T + seq2seq.decoder.b_fc
        
        best_word_idx = np.argmax(logits, axis=1)[0]
        if best_word_idx == 1: 
            break
            
        translated_words.append(urd_vocab.i2w[best_word_idx])
        current_word_idx = np.array([[best_word_idx]])
        
    return " ".join(translated_words)


for eng, _ in pairs[:5]:
    print(f"English: {eng}")
    print(f"Urdu:    {translate(eng)}\n")

In [63]:
translate('he is student')

'woh aik talib ilm hai'